# Module 5 Guided Lab: Basic Data Transformation with Pandas

**BAN 6003: Data Management and Analytics Integration**

In this lab, we use the `nycflights13` dataset to practice the core transformation moves that analysts use all the time:

- filter rows
- select columns
- sort rows
- rename columns
- create derived columns
- export a transformed dataset

This module is intentionally focused. We are **not** doing aggregation, reshaping, joins, or heavy validation this week. Those topics come later.

## Module 5 Learning Goals

By the end of this guided lab, you should be able to:

1. Filter observations using `query()`.
2. Sort rows using `sort_values()`.
3. Select columns using `[[...]]`, `.loc[]`, and `.filter()`.
4. Rename variables using `.rename()`.
5. Create derived columns using `.assign()`.
6. Explain how a transformation changes the meaning or usability of a dataset.
7. Export a transformed dataset for later work.

The main idea is simple: **transformation changes raw columns into more useful analytical variables.**

## Part 0: Setup

We will use the Python version of the classic `nycflights13` dataset. It contains flight records from New York City airports in 2013.

If you are using **GitHub Codespaces**, this package should already be installed from `requirements.txt` when the Codespace is created. If you are running locally and the import fails, install the packages from `requirements.txt` in your local environment, then rerun this notebook.


In [44]:
import nycflights13
import pandas as pd

print("Setup complete.")


Setup complete.


## Part 1: Load the Flights Dataset

The main table we will use is `flights`.

Each row represents one scheduled flight from a New York City airport in 2013. This row-level meaning matters. When we filter, sort, select, or create variables, we are still working at the **flight level**.

In [45]:
flights = nycflights13.flights.copy()

flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z


In [46]:
flights.shape

(336776, 19)

In [47]:
flights.columns

Index(['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay',
       'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight',
       'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute',
       'time_hour'],
      dtype='str')

### Quick orientation

Before transforming anything, take a quick look at the variables. Some important columns for this lab are:

- `month`, `day`: date components
- `dep_time`, `arr_time`: actual departure and arrival times
- `dep_delay`, `arr_delay`: departure and arrival delays in minutes
- `carrier`: airline carrier code
- `origin`, `dest`: origin and destination airport codes
- `air_time`: flight time in minutes
- `distance`: distance in miles

We are not cleaning these variables this week. We are using them to practice transformation grammar.

In [48]:
flights[["month", "day", "dep_delay", "arr_delay", "carrier", "origin", "dest", "air_time", "distance"]].head()

,month,day,dep_delay,arr_delay,carrier,origin,dest,air_time,distance
0,1,1,2.0,11.0,UA,EWR,IAH,227.0,1400
1,1,1,4.0,20.0,UA,LGA,IAH,227.0,1416
2,1,1,2.0,33.0,AA,JFK,MIA,160.0,1089
3,1,1,-1.0,-18.0,B6,JFK,BQN,183.0,1576
4,1,1,-6.0,-25.0,DL,LGA,ATL,116.0,762


## Part 2: Filtering Rows with `query()`

Filtering means keeping only the rows that meet a condition.

In business analytics, filtering is how we narrow the data to a relevant scenario. For example:

- flights in January
- flights to Houston
- flights that arrived more than two hours late
- flights that left on time but arrived late

The `query()` method lets us write filtering conditions in a readable way.

In [49]:
# Flights in January
january_flights = flights.query("month == 1")

january_flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z


In [50]:
# Flights on February 2
feb_2_flights = flights.query("month == 2 & day == 2")

feb_2_flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
112222,2013,2,2,3.0,2359,4.0,513.0,444,29.0,B6,739,N569JB,JFK,PSE,206.0,1617,23,59,2013-02-03T04:00:00Z
112223,2013,2,2,456.0,500,-4.0,640.0,648,-8.0,US,1117,N177US,EWR,CLT,84.0,529,5,0,2013-02-02T10:00:00Z
112224,2013,2,2,516.0,515,1.0,826.0,814,12.0,UA,785,N806UA,EWR,IAH,200.0,1400,5,15,2013-02-02T10:00:00Z
112225,2013,2,2,536.0,540,-4.0,838.0,850,-12.0,AA,1141,N638AA,JFK,MIA,159.0,1089,5,40,2013-02-02T10:00:00Z
112226,2013,2,2,543.0,530,13.0,835.0,829,6.0,UA,219,N490UA,LGA,IAH,212.0,1416,5,30,2013-02-02T10:00:00Z


In [51]:
# Flights on February 2 that departed late
feb_2_late_departures = flights.query("month == 2 & day == 2 & dep_delay > 0")

feb_2_late_departures.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
112222,2013,2,2,3.0,2359,4.0,513.0,444,29.0,B6,739,N569JB,JFK,PSE,206.0,1617,23,59,2013-02-03T04:00:00Z
112224,2013,2,2,516.0,515,1.0,826.0,814,12.0,UA,785,N806UA,EWR,IAH,200.0,1400,5,15,2013-02-02T10:00:00Z
112226,2013,2,2,543.0,530,13.0,835.0,829,6.0,UA,219,N490UA,LGA,IAH,212.0,1416,5,30,2013-02-02T10:00:00Z
112227,2013,2,2,549.0,540,9.0,1022.0,1017,5.0,B6,725,N558JB,JFK,BQN,196.0,1576,5,40,2013-02-02T10:00:00Z
112243,2013,2,2,601.0,600,1.0,930.0,927,3.0,UA,303,N532UA,JFK,SFO,371.0,2586,6,0,2013-02-02T11:00:00Z


### Common comparison operators in `query()`

| Operator | Meaning | Example |
|---|---|---|
| `==` | equals | `month == 1` |
| `!=` | not equal | `month != 12` |
| `>` | greater than | `arr_delay > 120` |
| `>=` | greater than or equal to | `arr_delay >= 120` |
| `<` | less than | `dep_delay < 0` |
| `<=` | less than or equal to | `dep_delay <= 0` |
| `in` | belongs to a list | `dest in ['IAH', 'HOU']` |

Use parentheses in your own thinking, even when Pandas does not require them, because filtering logic can get confusing quickly.

In [52]:
# Flights in November or December
holiday_season_flights = flights.query("month in [11, 12]")

holiday_season_flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
55893,2013,11,1,5.0,2359,6.0,352.0,345,7.0,B6,745,N568JB,JFK,PSE,205.0,1617,23,59,2013-11-02T03:00:00Z
55894,2013,11,1,35.0,2250,105.0,123.0,2356,87.0,B6,1816,N353JB,JFK,SYR,36.0,209,22,50,2013-11-02T02:00:00Z
55895,2013,11,1,455.0,500,-5.0,641.0,651,-10.0,US,1895,N192UW,EWR,CLT,88.0,529,5,0,2013-11-01T09:00:00Z
55896,2013,11,1,539.0,545,-6.0,856.0,827,29.0,UA,1714,N38727,LGA,IAH,229.0,1416,5,45,2013-11-01T09:00:00Z
55897,2013,11,1,542.0,545,-3.0,831.0,855,-24.0,AA,2243,N5CLAA,JFK,MIA,147.0,1089,5,45,2013-11-01T09:00:00Z


In [53]:
# Flights that arrived more than two hours late
very_late_arrivals = flights.query("arr_delay >= 120")

very_late_arrivals.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
119,2013,1,1,811.0,630,101.0,1047.0,830,137.0,MQ,4576,N531MQ,LGA,CLT,118.0,544,6,30,2013-01-01T11:00:00Z
151,2013,1,1,848.0,1835,853.0,1001.0,1950,851.0,MQ,3944,N942MQ,JFK,BWI,41.0,184,18,35,2013-01-01T23:00:00Z
218,2013,1,1,957.0,733,144.0,1056.0,853,123.0,UA,856,N534UA,EWR,BOS,37.0,200,7,33,2013-01-01T12:00:00Z
268,2013,1,1,1114.0,900,134.0,1447.0,1222,145.0,UA,1086,N76502,LGA,IAH,248.0,1416,9,0,2013-01-01T14:00:00Z
447,2013,1,1,1505.0,1310,115.0,1638.0,1431,127.0,EV,4497,N17984,EWR,RIC,63.0,277,13,10,2013-01-01T18:00:00Z


In [54]:
# Flights with missing tail numbers
missing_tailnum = flights.query("tailnum.isna()", engine="python")

missing_tailnum.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
1782,2013,1,2,NaN,1545,NaN,NaN,1910,NaN,AA,133,NaN,JFK,LAX,NaN,2475,15,45,2013-01-02T20:00:00Z
1784,2013,1,2,NaN,1601,NaN,NaN,1735,NaN,UA,623,NaN,EWR,ORD,NaN,719,16,1,2013-01-02T21:00:00Z
2697,2013,1,3,NaN,857,NaN,NaN,1209,NaN,UA,714,NaN,EWR,MIA,NaN,1085,8,57,2013-01-03T13:00:00Z
2698,2013,1,3,NaN,645,NaN,NaN,952,NaN,UA,719,NaN,EWR,DFW,NaN,1372,6,45,2013-01-03T11:00:00Z
3608,2013,1,4,NaN,845,NaN,NaN,1015,NaN,9E,3405,NaN,JFK,DCA,NaN,213,8,45,2013-01-04T13:00:00Z


### `query()` versus Boolean indexing

You may also see filtering written with Boolean indexing. This is equivalent to one of the examples above, but it is more verbose.

Both approaches are useful. In this course, `query()` is often easier to read for beginner-friendly filtering.

In [55]:
feb_2_late_departures_alt = flights[
    (flights["month"] == 2) &
    (flights["day"] == 2) &
    (flights["dep_delay"] > 0)
]

feb_2_late_departures_alt.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
112222,2013,2,2,3.0,2359,4.0,513.0,444,29.0,B6,739,N569JB,JFK,PSE,206.0,1617,23,59,2013-02-03T04:00:00Z
112224,2013,2,2,516.0,515,1.0,826.0,814,12.0,UA,785,N806UA,EWR,IAH,200.0,1400,5,15,2013-02-02T10:00:00Z
112226,2013,2,2,543.0,530,13.0,835.0,829,6.0,UA,219,N490UA,LGA,IAH,212.0,1416,5,30,2013-02-02T10:00:00Z
112227,2013,2,2,549.0,540,9.0,1022.0,1017,5.0,B6,725,N558JB,JFK,BQN,196.0,1576,5,40,2013-02-02T10:00:00Z
112243,2013,2,2,601.0,600,1.0,930.0,927,3.0,UA,303,N532UA,JFK,SFO,371.0,2586,6,0,2013-02-02T11:00:00Z


### Your Turn 1: Filtering

Use `query()` to answer the following. For each task, show the number of matching rows using `.shape[0]`.

1. How many flights had an arrival delay of two or more hours?
2. How many flights flew to Houston, meaning destination is either `IAH` or `HOU`?
3. How many flights arrived more than two hours late but did not leave late?

In [56]:
# 1. Flights with arrival delay of two or more hours
flights.query("arr_delay >= 120").shape[0]

10200

In [57]:
# 2. Flights to Houston: IAH or HOU
flights.query("dest in ['IAH', 'HOU']").shape[0]

9313

In [58]:
# 3. Flights that arrived more than two hours late but did not leave late
flights.query("arr_delay > 120 & dep_delay <= 0").shape[0]

29

## Part 3: Sorting Rows with `sort_values()`

Sorting does not remove rows. It changes the order so that important cases rise to the top.

This is useful when you want to inspect extremes, such as:

- the most delayed flights
- the earliest departures
- the longest flights
- the shortest flights

Sorting is a transformation because it changes how we review and interpret the data.

In [59]:
# Flights with the smallest departure delays first
flights.sort_values("dep_delay").head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
89673,2013,12,7,2040.0,2123,-43.0,40.0,2352,48.0,B6,97,N592JB,JFK,DEN,265.0,1626,21,23,2013-12-08T02:00:00Z
113633,2013,2,3,2022.0,2055,-33.0,2240.0,2338,-58.0,DL,1715,N612DL,LGA,MSY,162.0,1183,20,55,2013-02-04T01:00:00Z
64501,2013,11,10,1408.0,1440,-32.0,1549.0,1559,-10.0,EV,5713,N825AS,LGA,IAD,52.0,229,14,40,2013-11-10T19:00:00Z
9619,2013,1,11,1900.0,1930,-30.0,2233.0,2243,-10.0,DL,1435,N934DL,LGA,TPA,139.0,1010,19,30,2013-01-12T00:00:00Z
24915,2013,1,29,1703.0,1730,-27.0,1947.0,1957,-10.0,F9,837,N208FR,LGA,DEN,250.0,1620,17,30,2013-01-29T22:00:00Z


In [60]:
# Flights with the largest departure delays first
flights.sort_values("dep_delay", ascending=False).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
7072,2013,1,9,641.0,900,1301.0,1242.0,1530,1272.0,HA,51,N384HA,JFK,HNL,640.0,4983,9,0,2013-01-09T14:00:00Z
235778,2013,6,15,1432.0,1935,1137.0,1607.0,2120,1127.0,MQ,3535,N504MQ,JFK,CMH,74.0,483,19,35,2013-06-15T23:00:00Z
8239,2013,1,10,1121.0,1635,1126.0,1239.0,1810,1109.0,MQ,3695,N517MQ,EWR,ORD,111.0,719,16,35,2013-01-10T21:00:00Z
327043,2013,9,20,1139.0,1845,1014.0,1457.0,2210,1007.0,AA,177,N338AA,JFK,SFO,354.0,2586,18,45,2013-09-20T22:00:00Z
270376,2013,7,22,845.0,1600,1005.0,1044.0,1815,989.0,MQ,3075,N665MQ,JFK,CVG,96.0,589,16,0,2013-07-22T20:00:00Z


In [61]:
# Sort by multiple variables
# First by departure delay, then by arrival delay
flights.sort_values(["dep_delay", "arr_delay"]).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
89673,2013,12,7,2040.0,2123,-43.0,40.0,2352,48.0,B6,97,N592JB,JFK,DEN,265.0,1626,21,23,2013-12-08T02:00:00Z
113633,2013,2,3,2022.0,2055,-33.0,2240.0,2338,-58.0,DL,1715,N612DL,LGA,MSY,162.0,1183,20,55,2013-02-04T01:00:00Z
64501,2013,11,10,1408.0,1440,-32.0,1549.0,1559,-10.0,EV,5713,N825AS,LGA,IAD,52.0,229,14,40,2013-11-10T19:00:00Z
9619,2013,1,11,1900.0,1930,-30.0,2233.0,2243,-10.0,DL,1435,N934DL,LGA,TPA,139.0,1010,19,30,2013-01-12T00:00:00Z
24915,2013,1,29,1703.0,1730,-27.0,1947.0,1957,-10.0,F9,837,N208FR,LGA,DEN,250.0,1620,17,30,2013-01-29T22:00:00Z


In [62]:
# Mixed sort directions:
# departure delay descending, arrival delay ascending
flights.sort_values(["dep_delay", "arr_delay"], ascending=[False, True]).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
7072,2013,1,9,641.0,900,1301.0,1242.0,1530,1272.0,HA,51,N384HA,JFK,HNL,640.0,4983,9,0,2013-01-09T14:00:00Z
235778,2013,6,15,1432.0,1935,1137.0,1607.0,2120,1127.0,MQ,3535,N504MQ,JFK,CMH,74.0,483,19,35,2013-06-15T23:00:00Z
8239,2013,1,10,1121.0,1635,1126.0,1239.0,1810,1109.0,MQ,3695,N517MQ,EWR,ORD,111.0,719,16,35,2013-01-10T21:00:00Z
327043,2013,9,20,1139.0,1845,1014.0,1457.0,2210,1007.0,AA,177,N338AA,JFK,SFO,354.0,2586,18,45,2013-09-20T22:00:00Z
270376,2013,7,22,845.0,1600,1005.0,1044.0,1815,989.0,MQ,3075,N665MQ,JFK,CVG,96.0,589,16,0,2013-07-22T20:00:00Z


### Your Turn 2: Sorting

Use `sort_values()` to answer the following:

1. Which flights had the largest departure delays?
2. Which flights left earliest based on `dep_time`?
3. Which flights traveled the longest distance?
4. Which flights traveled the shortest distance?

Use `.head()` so the output is manageable.

In [63]:
# 1. Largest departure delays
flights.sort_values("dep_delay", ascending=False).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
7072,2013,1,9,641.0,900,1301.0,1242.0,1530,1272.0,HA,51,N384HA,JFK,HNL,640.0,4983,9,0,2013-01-09T14:00:00Z
235778,2013,6,15,1432.0,1935,1137.0,1607.0,2120,1127.0,MQ,3535,N504MQ,JFK,CMH,74.0,483,19,35,2013-06-15T23:00:00Z
8239,2013,1,10,1121.0,1635,1126.0,1239.0,1810,1109.0,MQ,3695,N517MQ,EWR,ORD,111.0,719,16,35,2013-01-10T21:00:00Z
327043,2013,9,20,1139.0,1845,1014.0,1457.0,2210,1007.0,AA,177,N338AA,JFK,SFO,354.0,2586,18,45,2013-09-20T22:00:00Z
270376,2013,7,22,845.0,1600,1005.0,1044.0,1815,989.0,MQ,3075,N665MQ,JFK,CVG,96.0,589,16,0,2013-07-22T20:00:00Z


In [64]:
# 2. Earliest departure times
flights.sort_values("dep_time").head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
10452,2013,1,13,1.0,2249,72.0,108.0,2357,71.0,B6,22,N206JB,JFK,SYR,41.0,209,22,49,2013-01-14T03:00:00Z
100800,2013,12,20,1.0,2359,2.0,430.0,440,-10.0,B6,1503,N608JB,JFK,SJU,182.0,1598,23,59,2013-12-21T04:00:00Z
212954,2013,5,22,1.0,1935,266.0,154.0,2140,254.0,EV,4361,N27200,EWR,TYS,94.0,631,19,35,2013-05-22T23:00:00Z
297030,2013,8,19,1.0,2359,2.0,347.0,350,-3.0,B6,745,N552JB,JFK,PSE,201.0,1617,23,59,2013-08-20T03:00:00Z
131559,2013,2,24,1.0,2245,76.0,121.0,2354,87.0,B6,608,N216JB,JFK,PWM,56.0,273,22,45,2013-02-25T03:00:00Z


In [65]:
# 3. Longest distances
flights.sort_values("distance", ascending=False).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
118311,2013,2,9,1206.0,900,186.0,1814.0,1540,154.0,HA,51,N380HA,JFK,HNL,645.0,4983,9,0,2013-02-09T14:00:00Z
227103,2013,6,6,1044.0,1000,44.0,1441.0,1435,6.0,HA,51,N384HA,JFK,HNL,580.0,4983,10,0,2013-06-06T14:00:00Z
224146,2013,6,3,957.0,1000,-3.0,1432.0,1435,-3.0,HA,51,N381HA,JFK,HNL,605.0,4983,10,0,2013-06-03T14:00:00Z
170226,2013,4,6,957.0,1000,-3.0,1510.0,1510,0.0,HA,51,N382HA,JFK,HNL,638.0,4983,10,0,2013-04-06T14:00:00Z
130088,2013,2,22,857.0,900,-3.0,1436.0,1540,-64.0,HA,51,N382HA,JFK,HNL,606.0,4983,9,0,2013-02-22T14:00:00Z


In [66]:
# 4. Shortest distances
flights.sort_values("distance").head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
275945,2013,7,27,NaN,106,NaN,NaN,245,NaN,US,1632,NaN,EWR,LGA,NaN,17,1,6,2013-07-27T05:00:00Z
116412,2013,2,6,2125.0,2129,-4.0,2207.0,2224,-17.0,EV,4619,N12564,EWR,PHL,28.0,80,21,29,2013-02-07T02:00:00Z
164109,2013,3,30,1942.0,1950,-8.0,2026.0,2044,-18.0,EV,4457,N12569,EWR,PHL,24.0,80,19,50,2013-03-30T23:00:00Z
150994,2013,3,16,1947.0,1950,-3.0,2055.0,2044,11.0,EV,4457,N48901,EWR,PHL,31.0,80,19,50,2013-03-16T23:00:00Z
19088,2013,1,22,2203.0,2130,33.0,2317.0,2225,52.0,EV,4619,N13538,EWR,PHL,31.0,80,21,30,2013-01-23T02:00:00Z


## Part 4: Selecting Columns

Selecting columns means keeping only the variables you need.

This is an important transformation habit because real datasets often have many columns. A smaller working DataFrame is easier to read, easier to debug, and easier to explain.

### Single brackets versus double brackets

A very common beginner issue is the difference between selecting one column as a Series and selecting one column as a DataFrame.

In [67]:
one_column_series = flights["year"]

type(one_column_series)

pandas.Series

In [68]:
one_column_dataframe = flights[["year"]]

type(one_column_dataframe)

pandas.DataFrame

### Selecting multiple columns

To select multiple columns, use a list of column names inside double brackets.

In [69]:
date_delay_columns = flights[["year", "month", "day", "dep_delay", "arr_delay"]]

date_delay_columns.head()

,year,month,day,dep_delay,arr_delay
0,2013,1,1,2.0,11.0
1,2013,1,1,4.0,20.0
2,2013,1,1,2.0,33.0
3,2013,1,1,-1.0,-18.0
4,2013,1,1,-6.0,-25.0


In [70]:
# Equivalent selection using .loc[row_selection, column_selection]
date_delay_columns_loc = flights.loc[:, ["year", "month", "day", "dep_delay", "arr_delay"]]

date_delay_columns_loc.head()

,year,month,day,dep_delay,arr_delay
0,2013,1,1,2.0,11.0
1,2013,1,1,4.0,20.0
2,2013,1,1,2.0,33.0
3,2013,1,1,-1.0,-18.0
4,2013,1,1,-6.0,-25.0


### Finding columns by keyword

The `.filter(like=...)` method is useful when column names share a pattern.

For example, many variables related to departure include `"dep"` in the name.

In [71]:
flights.filter(like="dep").head()

,dep_time,sched_dep_time,dep_delay
0,517.0,515,2.0
1,533.0,529,4.0
2,542.0,540,2.0
3,544.0,545,-1.0
4,554.0,600,-6.0


### Selecting from a requested list safely

Sometimes you have a list of desired columns, but not every name exists in the DataFrame. The `.columns.intersection()` approach keeps only names that actually exist.

This is useful when you are working with changing datasets or user-provided column lists.

In [72]:
requested_columns = ["MONTH", "month", "day", "dep_delay", "arr_delay"]

available_columns = flights.columns.intersection(requested_columns)

available_columns

Index(['month', 'day', 'dep_delay', 'arr_delay'], dtype='str')

In [73]:
flights[available_columns].head()

,month,day,dep_delay,arr_delay
0,1,1,2.0,11.0
1,1,1,4.0,20.0
2,1,1,2.0,33.0
3,1,1,-1.0,-18.0
4,1,1,-6.0,-25.0


### Your Turn 3: Selecting Columns

1. What happens if you include the same variable more than once inside double brackets?
2. Select the columns included in this list, but only if they exist in the dataset:

```python
var_list = ["MONTH", "month", "day", "dep_delay", "arr_delay"]
```

3. Select all columns that contain `"time"` in the name.

In [74]:
# 1. Include a variable multiple times
flights[["month", "month"]].head()

,month,month
0,1,1
1,1,1
2,1,1
3,1,1
4,1,1


In [75]:
# 2. Select only available variables from the list
var_list = ["MONTH", "month", "day", "dep_delay", "arr_delay"]

available_vars = flights.columns.intersection(var_list)
flights[available_vars].head()

,month,day,dep_delay,arr_delay
0,1,1,2.0,11.0
1,1,1,4.0,20.0
2,1,1,2.0,33.0
3,1,1,-1.0,-18.0
4,1,1,-6.0,-25.0


In [76]:
# 3. Select variables that contain "time" in the name
flights.filter(like="time").head()

,dep_time,sched_dep_time,arr_time,sched_arr_time,air_time,time_hour
0,517.0,515,830.0,819,227.0,2013-01-01T10:00:00Z
1,533.0,529,850.0,830,227.0,2013-01-01T10:00:00Z
2,542.0,540,923.0,850,160.0,2013-01-01T10:00:00Z
3,544.0,545,1004.0,1022,183.0,2013-01-01T10:00:00Z
4,554.0,600,812.0,837,116.0,2013-01-01T11:00:00Z


## Part 5: Renaming Columns

Renaming is a small change that can make a dataset much easier to understand.

In practice, raw variable names are often abbreviated or inconsistent. We do not always rename every variable, but we often rename key variables to make later notebooks and reports clearer.

In [77]:
flight_sml = flights[["year", "month", "day", "arr_delay", "dep_delay", "air_time", "distance"]].copy()

flight_sml.head()

,year,month,day,arr_delay,dep_delay,air_time,distance
0,2013,1,1,11.0,2.0,227.0,1400
1,2013,1,1,20.0,4.0,227.0,1416
2,2013,1,1,33.0,2.0,160.0,1089
3,2013,1,1,-18.0,-1.0,183.0,1576
4,2013,1,1,-25.0,-6.0,116.0,762


In [78]:
flight_sml = flight_sml.rename(columns={
    "arr_delay": "arrival_delay",
    "dep_delay": "departure_delay"
})

flight_sml.head()

,year,month,day,arrival_delay,departure_delay,air_time,distance
0,2013,1,1,11.0,2.0,227.0,1400
1,2013,1,1,20.0,4.0,227.0,1416
2,2013,1,1,33.0,2.0,160.0,1089
3,2013,1,1,-18.0,-1.0,183.0,1576
4,2013,1,1,-25.0,-6.0,116.0,762


### Naming tip

Use clear `snake_case` names for new columns:

- good: `arrival_delay`, `departure_delay`, `speed_mph`
- harder to read: `arrDelay`, `x1`, `newvar`, `delay2`

Good names help future you, your teammates, and your instructor understand your work.

## Part 6: Creating Derived Columns with `assign()`

Derived columns are new variables created from existing variables.

This is one of the most important transformation skills in analytics. Derived variables often become features for modeling, KPIs for reporting, or business indicators for decision-making.

For this lab, we will create:

- `delay_change_minutes`: arrival delay minus departure delay; positive values mean the flight became more delayed, while negative values mean it recovered time
- `air_time_hours`: air time converted from minutes to hours
- `speed_mph`: distance divided by air time in hours

The lecture used the shorter label `delay_gain` for the same arrival-minus-departure calculation. Here we use `delay_change_minutes` so the direction and unit are explicit.

In [79]:
flight_sml = flights[["year", "month", "day", "arr_delay", "dep_delay", "air_time", "distance"]].copy()

flight_sml = flight_sml.assign(
    delay_change_minutes = flight_sml["arr_delay"] - flight_sml["dep_delay"],
    air_time_hours = flight_sml["air_time"] / 60
)

flight_sml.head()

,year,month,day,arr_delay,dep_delay,air_time,distance,delay_change_minutes,air_time_hours
0,2013,1,1,11.0,2.0,227.0,1400,9.0,3.783333
1,2013,1,1,20.0,4.0,227.0,1416,16.0,3.783333
2,2013,1,1,33.0,2.0,160.0,1089,31.0,2.666667
3,2013,1,1,-18.0,-1.0,183.0,1576,-17.0,3.050000
4,2013,1,1,-25.0,-6.0,116.0,762,-19.0,1.933333


In [80]:
flight_sml = flight_sml.assign(
    speed_mph = flight_sml["distance"] / flight_sml["air_time_hours"]
)

flight_sml.head()

,year,month,day,arr_delay,dep_delay,air_time,distance,delay_change_minutes,air_time_hours,speed_mph
0,2013,1,1,11.0,2.0,227.0,1400,9.0,3.783333,370.044053
1,2013,1,1,20.0,4.0,227.0,1416,16.0,3.783333,374.273128
2,2013,1,1,33.0,2.0,160.0,1089,31.0,2.666667,408.375000
3,2013,1,1,-18.0,-1.0,183.0,1576,-17.0,3.050000,516.721311
4,2013,1,1,-25.0,-6.0,116.0,762,-19.0,1.933333,394.137931


### Your Turn 4: Creating Derived Columns

Create a smaller DataFrame called `flight_efficiency` that contains:

- `air_time`
- `distance`

Then create:

1. `distance_km`, converting distance from miles to kilometers using `1 mile = 1.61 kilometers`.
2. `minutes_per_km`, using `air_time / distance_km`.

Show the first few rows.

In [81]:
# Create flight_efficiency with air_time and distance
flight_efficiency = flights[["air_time", "distance"]].copy()

flight_efficiency.head()

,air_time,distance
0,227.0,1400
1,227.0,1416
2,160.0,1089
3,183.0,1576
4,116.0,762


In [82]:
# Add distance_km
flight_efficiency = flight_efficiency.assign(
    distance_km = flight_efficiency["distance"] * 1.61
)

flight_efficiency.head()

,air_time,distance,distance_km
0,227.0,1400,2254.00
1,227.0,1416,2279.76
2,160.0,1089,1753.29
3,183.0,1576,2537.36
4,116.0,762,1226.82


In [83]:
# Add minutes_per_km
flight_efficiency = flight_efficiency.assign(
    minutes_per_km = flight_efficiency["air_time"] / flight_efficiency["distance_km"]
)

flight_efficiency.head()

,air_time,distance,distance_km,minutes_per_km
0,227.0,1400,2254.00,0.100710
1,227.0,1416,2279.76,0.099572
2,160.0,1089,1753.29,0.091257
3,183.0,1576,2537.36,0.072122
4,116.0,762,1226.82,0.094553


## Part 7: Put the Transformation Steps Together

Scenario: an operations analyst needs a reusable row-level file for flights departing JFK during June, July, and August. The file should support later delay review without introducing aggregation yet.

We will:

1. filter to the requested flights and require usable air time and distance;
2. select and rename relevant columns;
3. create clearly defined derived columns;
4. sort the most delayed arrivals first; and
5. export the transformed dataset.

Each row will still represent one flight.

In [84]:
# Step 1: filter to the business scope and usable flight records
transformed_flights = flights.query(
    "origin == 'JFK' & month in [6, 7, 8] & air_time.notna() & distance.notna()",
    engine="python",
)

# Step 2: select useful columns
transformed_flights = transformed_flights[[
    "year", "month", "day", "carrier", "origin", "dest",
    "dep_delay", "arr_delay", "air_time", "distance"
]]

# Step 3: rename columns for clarity
transformed_flights = transformed_flights.rename(columns={
    "dep_delay": "departure_delay",
    "arr_delay": "arrival_delay"
})

# Step 4: create derived columns with explicit units and direction
transformed_flights = transformed_flights.assign(
    delay_change_minutes=(
        transformed_flights["arrival_delay"] - transformed_flights["departure_delay"]
    ),
    air_time_hours=transformed_flights["air_time"] / 60,
    speed_mph=(
        transformed_flights["distance"] / (transformed_flights["air_time"] / 60)
    ),
)

# Step 5: put the most delayed arrivals first
transformed_flights = transformed_flights.sort_values(
    "arrival_delay", ascending=False
).reset_index(drop=True)

transformed_flights.head()

,year,month,day,carrier,origin,dest,departure_delay,arrival_delay,air_time,distance,delay_change_minutes,air_time_hours,speed_mph
0,2013,6,15,MQ,JFK,CMH,1137.0,1127.0,74.0,483,-10.0,1.233333,391.621622
1,2013,7,22,MQ,JFK,CVG,1005.0,989.0,96.0,589,-16.0,1.600000,368.125000
2,2013,6,27,DL,JFK,PDX,899.0,850.0,313.0,2454,-49.0,5.216667,470.415335
3,2013,6,27,DL,JFK,SAN,790.0,769.0,312.0,2446,-21.0,5.200000,470.384615
4,2013,7,7,VX,JFK,SFO,629.0,676.0,334.0,2586,47.0,5.566667,464.550898


### Short Reflection

Write 3-4 sentences below.

1. What does one row in `transformed_flights` represent, and which flights are excluded by the business filter?
2. What does a positive versus negative `delay_change_minutes` value mean?
3. Why is `speed_mph` only an approximate airborne speed?

This is not a full transformation summary yet. For now, practice explaining what the transformed file can and cannot support.

**Your response here:**

- Each row in transformed_flights represents one flight departing from JFK during June, July, or August with usable air time and distance data. Flights outside those months, flights not departing from JFK, and flights missing air time or distance are excluded. 
- A positive delay_change_minutes value means the flight became more delayed during the trip, while a negative value means it made up some time. 
- Speed_mph is only an approximate airborne speed because it uses total distance divided by air time so it represents an average speed rather than the plane's exact speed throughout the flight.

## Part 8: Export the Transformed Dataset

Exporting gives us a saved output that can be reused later.

For now, we will export a CSV file. Later in the course, we will discuss other formats and database storage.

In [85]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "flights_transformed_module5.csv"
transformed_flights.to_csv(output_path, index=False)

print(f"Saved {len(transformed_flights):,} rows to: {output_path}")

Saved 28,809 rows to: ../data/processed/flights_transformed_module5.csv


## Part 9: Offline Assignment - Operational Delay Review

Complete this task independently using `transformed_flights`.

An operations manager wants a compact review file containing the 50 JFK summer flights with the largest arrival delays among flights delayed by at least 60 minutes.

1. Filter `transformed_flights` to `arrival_delay >= 60`.
2. Select `year`, `month`, `day`, `carrier`, `dest`, `departure_delay`, `arrival_delay`, and `delay_change_minutes`.
3. Sort by `arrival_delay` from largest to smallest and keep the first 50 rows.
4. Save the result as `data/processed/jfk_summer_delay_review_top50.csv`.
5. Display the first five rows and the final shape.

In [86]:
# Offline Assignment - Your code here
# Suggested sequence:
# 1. Use .query() for the delay condition.
# 2. Select the eight requested columns with [[...]].
# 3. Use .sort_values(..., ascending=False).head(50).
# 4. Save with output_dir / "jfk_summer_delay_review_top50.csv".
# Build a DataFrame named jfk_summer_delay_review.
jfk_summer_delay_review = transformed_flights.query(
    "arrival_delay >= 60"
)

jfk_summer_delay_review = jfk_summer_delay_review[[
    "year", "month", "day", "carrier", "dest",
    "departure_delay", "arrival_delay", "delay_change_minutes"
]]

jfk_summer_delay_review = jfk_summer_delay_review.sort_values(
    "arrival_delay", ascending=False
).head(50)

output_dir = Path("../data/processed")
jfk_summer_delay_review.to_csv(
    output_dir / "jfk_summer_delay_review_top50.csv",
    index=False
)

jfk_summer_delay_review.head()
jfk_summer_delay_review.shape


(50, 8)

### Offline Assignment Reflection

**Your response here:** Write 4-5 sentences explaining what one row represents, why this file is not representative of all flights, what `delay_change_minutes` adds to the review, and one question the data cannot answer by itself.

Each row represents one JFK summer flight that had an arrival delay of at least 60 minutes. This file does not represent all flights because it only includes the 50 flights with the largest arrival delays from the filtered data. The delay_change_minutes column shows whether a flight became more or less delayed during the trip. This helps us see how the delay changed instead of only looking at the final arrival delay. The data cannot tell us why a specific flight was delayed.

## Lab Wrap-Up

In this lab, you practiced the core Pandas transformation moves:

- `query()` for filtering rows
- `sort_values()` for ordering rows
- `[[...]]`, `.loc[]`, and `.filter()` for selecting columns
- `rename()` for clearer variable names
- `assign()` for derived columns
- `to_csv()` for exporting a transformed dataset

Before submitting, complete the offline assignment, rerun the notebook from the top, and confirm that both requested outputs are in `data/processed`.

Next module, we will focus on **string and categorical data transformation**.